# RT-04: Introducción a Sensores IoT



## 📋 Contexto del Caso de Negocio

**Empresa:** "DistribuTech Logistics" - Empresa de logística especializada en distribución urbana con flota de 150+ vehículos equipados con sensores IoT.

**Situación actual:**
- **Flota equipada**: 150+ vehículos con sensores IoT (GPS, temperatura, estado de motor)
- **Problema:** Datos de sensores sin analizar, pérdida de insights sobre comportamiento de flota
- Factores relevantes:
  - Sensores generan ~5,000 eventos/día con datos de ubicación, temperatura y estados
  - Falta de visibilidad en tiempo real sobre ruta y condiciones de transporte
  - Sin sistema de alertas automáticas para eventos críticos

**Impacto financiero:**
- Pérdidas por deterioro de producto: ~$50K/mes (falta de monitoreo de temperatura)
- Costes de combustible sin optimizar: ~$30K/mes (rutas no monitoreadas)
- Tiempo de respuesta a incidentes: 4+ horas (sin alertas en tiempo real)

**Objetivo:** Implementar análisis exploratorio de datos IoT para:
1. Comprender estructura y patrones de datos de sensores
2. Identificar eventos críticos y alertas
3. Visualizar trazabilidad geográfica de la flota
4. Establecer baseline de KPIs de monitoreo

### 💼 ¿Por qué es IMPORTANTE?
- **Visibilidad Operativa:** Monitoreo de flota en tiempo real permite toma de decisiones rápida
- **Reducción de Pérdidas:** Detección temprana de excursiones de temperatura reduce deterioro
- **Optimización de Costes:** Análisis de patrones de ruta identifica oportunidades de ahorro
- **Mejora de Servicio:** Trazabilidad GPS mejora tiempos de respuesta a clientes

### 🎁 ¿PARA QUÉ sirve?
- **Análisis Exploratorio:** Entender tipos de sensores, frecuencia de datos y patrones
- **Dashboard de KPIs:** Establecer métricas de monitoreo de flota (delivery rate, eventos/orden)
- **Detección de Anomalías:** Identificar eventos críticos y alertas automáticas
- **Base para ML:** Sentar fundamentos para modelos predictivos de mantenimiento

### 🔧 ¿CÓMO se implementa?
- **Datos requeridos:** `transport_events.csv`, `locations.csv`
- **Técnica principal:** Análisis exploratorio de series temporales IoT, geolocalización
- **Métricas resultado:** 
  - `Delivery Rate = (Órdenes Entregadas / Total Órdenes) × 100`
  - `Eventos Promedio/Orden = Total Eventos / Total Órdenes`
- **Técnica aplicada:** EDA de IoT, visualización geoespacial con Plotly, análisis de eventos

---

## 🎯 Contexto del Notebook

### ¿Qué?
Este notebook es una introducción al análisis de datos IoT generados por sensores en una red de distribución logística.

### ¿Por qué?
Los datos de sensores (GPS, temperatura, estados) se generan continuamente pero sin análisis representan oportunidad perdida de optimización y prevención de problemas.

### ¿Para qué?
- Establecer baseline de KPIs de monitoreo de flota
- Identificar patrones en eventos de transporte
- Visualizar trazabilidad geográfica de órdenes
- Detectar eventos críticos y alertas
- Sentar bases para análisis avanzados (streaming, ML)

### ¿Cuándo?
- Como primer paso antes de implementar sistemas de monitoreo en tiempo real
- Para evaluar despliegue de nueva red de sensores
- Al revisar performance histórica de flota
- Como contenido educativo sobre arquitecturas IoT

### ¿Cómo?
1. Cargar datos históricos de eventos de transporte
2. Analizar distribución y patrones temporales
3. Visualizar rutas geográficas con mapas interactivos
4. Calcular KPIs de delivery y monitoreo
5. Identificar eventos y órdenes críticas

---

## 🎯 Objetivos de Aprendizaje

- Entender la estructura de datos de sensores IoT.
- Simular flujos de datos de sensores industriales.
- Visualizar series temporales de alta frecuencia.


## 📦 Instalación de Librerías Necesarias

**Antes de ejecutar este notebook, asegúrate de tener instaladas todas las dependencias.**

### Opción 1: Instalación dentro del notebook
Ejecuta la siguiente celda para instalar las librerías necesarias:

```python
%pip install pandas numpy plotly
```

### Opción 2: Instalación desde terminal
Si prefieres instalar desde la terminal, ejecuta:

```bash
# PowerShell o CMD
pip install pandas numpy plotly

# O si usas el proyecto completo con pyproject.toml
pip install -e .[core,notebooks,iot]
```

### Librerías requeridas:
- `pandas`: Manipulación y análisis de datos de sensores
- `numpy`: Cálculos numéricos y simulación de datos
- `plotly`: Visualización interactiva de mapas y series temporales
- `datetime`: Manejo de timestamps de eventos IoT

---

### 📝 Información del Notebook

| Campo | Valor |
| :--- | :--- |
| **🆔 ID** | `RT-04` |
| **📛 Título** | `IoT Sensors Intro` |
| **🔹 Especialidad** | `IoT / Real-Time Analytics` |
| **⚙️ Proceso** | `Deliver / Monitor` |
| **🧠 Nivel** | `Basic` |
| **⏱️ Duración** | `25 min` |
| **🏷️ Etiquetas** | `iot`, `sensores`, `monitoreo`, `telemetría`, `fleet-tracking` |

---

## ⚙️ Configuración Inicial

In [17]:
# ⚙️ Configuración de rutas
import sys
from pathlib import Path

def resolve_repo_root():
    """Detecta raíz del repositorio buscando carpetas data/ y notebooks/"""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / 'data').exists() and (parent / 'notebooks').exists():
            return parent
    return current

root = resolve_repo_root()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"✅ Rutas configuradas: {root}")

✅ Rutas configuradas: f:\GitHub\supply-chain-data-notebooks


In [18]:
# 📚 Importar librerías
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Rutas
DATA_DIR = root / "data" / "raw"
OUTPUT_DIR = root / "data" / "processed"
OUTPUT_DIR.mkdir(exist_ok=True)

print("✅ Librerías cargadas")
print(f"📁 Directorio datos: {DATA_DIR.resolve()}")

✅ Librerías cargadas
📁 Directorio datos: F:\GitHub\supply-chain-data-notebooks\data\raw


---

# 🔧 PASOS DEL NOTEBOOK

---

## 📂 Paso 1: Cargar Datos de Sensores

In [20]:
# Generar datos sintéticos realistas de sensores IoT si no existen
import os

transport_file = DATA_DIR / "transport_events.csv"
locations_file = DATA_DIR / "locations.csv"

if not transport_file.exists() or not locations_file.exists():
    print("⚙️ Generando datos sintéticos realistas de sensores IoT...\n")
    
    # Seed para reproducibilidad
    np.random.seed(42)
    
    # Generar ubicaciones realistas (almacenes y puntos de entrega en área urbana)
    n_locations = 15
    # Coordenadas base Ciudad de México
    base_lat, base_lon = 19.4326, -99.1332
    
    locations_data = {
        'location_id': [f'LOC_{i:03d}' for i in range(n_locations)],
        'location_type': np.random.choice(['WAREHOUSE', 'DISTRIBUTION_CENTER', 'CUSTOMER'], 
                                         n_locations, p=[0.2, 0.1, 0.7]),
        'lat': base_lat + np.random.randn(n_locations) * 0.15,
        'lon': base_lon + np.random.randn(n_locations) * 0.15,
        'name': [f'Ubicación {i}' for i in range(n_locations)]
    }
    df_locations = pd.DataFrame(locations_data)
    
    # Generar eventos de transporte realistas
    n_orders = 50
    events_list = []
    
    # Estados típicos de IoT en logística
    status_flow = ['CREATED', 'DISPATCHED', 'IN_TRANSIT', 'DELIVERED']
    
    for order_num in range(n_orders):
        order_id = f'ORD_{order_num:04d}'
        
        # Timestamp inicial (últimos 7 días)
        start_time = pd.Timestamp.now() - pd.Timedelta(days=np.random.randint(1, 8))
        
        # Número de eventos por orden (realista: 3-8 eventos)
        n_events = np.random.randint(3, 9)
        
        # Seleccionar ubicación origen y destino
        origin_idx = np.random.randint(0, n_locations)
        dest_idx = np.random.randint(0, n_locations)
        while dest_idx == origin_idx:
            dest_idx = np.random.randint(0, n_locations)
        
        current_lat = df_locations.iloc[origin_idx]['lat']
        current_lon = df_locations.iloc[origin_idx]['lon']
        target_lat = df_locations.iloc[dest_idx]['lat']
        target_lon = df_locations.iloc[dest_idx]['lon']
        
        # Generar eventos de la orden
        for event_idx in range(n_events):
            # Estado progresivo
            if event_idx == 0:
                status = 'CREATED'
            elif event_idx < n_events * 0.3:
                status = 'DISPATCHED'
            elif event_idx < n_events * 0.8:
                status = 'IN_TRANSIT'
            else:
                status = 'DELIVERED'
            
            # Timestamp incremental (1-4 horas entre eventos)
            timestamp = start_time + pd.Timedelta(hours=event_idx * np.random.uniform(1, 4))
            
            # Interpolación de coordenadas (movimiento progresivo hacia destino)
            progress = event_idx / (n_events - 1)
            event_lat = current_lat + (target_lat - current_lat) * progress + np.random.randn() * 0.01
            event_lon = current_lon + (target_lon - current_lon) * progress + np.random.randn() * 0.01
            
            # Telemetría realista de sensores
            temperature = np.random.uniform(2, 8) if status != 'DELIVERED' else np.random.uniform(4, 6)
            humidity = np.random.uniform(60, 80)
            battery = max(20, 100 - event_idx * np.random.uniform(5, 15))
            
            events_list.append({
                'event_id': f'EVT_{order_num:04d}_{event_idx:02d}',
                'order_id': order_id,
                'timestamp': timestamp,
                'status': status,
                'lat': event_lat,
                'lon': event_lon,
                'temperature_c': round(temperature, 1),
                'humidity_pct': round(humidity, 1),
                'battery_pct': round(battery, 1),
                'signal_strength': np.random.randint(-90, -40)
            })
    
    df_transport = pd.DataFrame(events_list)
    
    # Guardar datasets
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    df_transport.to_csv(transport_file, index=False)
    df_locations.to_csv(locations_file, index=False)
    print("✅ Datasets sintéticos generados y guardados\n")
else:
    # Cargar datasets existentes
    df_transport = pd.read_csv(transport_file, parse_dates=['timestamp'])
    df_locations = pd.read_csv(locations_file)

print("📡 Datos de Sensores IoT:")
print(f"  - Eventos: {len(df_transport):,} registros")
print(f"  - Órdenes únicas: {df_transport['order_id'].nunique()} órdenes")
print(f"  - Ubicaciones: {len(df_locations)} locations")
print(f"  - Rango temporal: {df_transport['timestamp'].min()} a {df_transport['timestamp'].max()}")
print(f"  - Columnas telemetría: {[col for col in df_transport.columns if col in ['temperature_c', 'humidity_pct', 'battery_pct', 'signal_strength']]}")

display(df_transport.head(10))

⚙️ Generando datos sintéticos realistas de sensores IoT...

✅ Datasets sintéticos generados y guardados

📡 Datos de Sensores IoT:
  - Eventos: 270 registros
  - Órdenes únicas: 50 órdenes
  - Ubicaciones: 15 locations
  - Rango temporal: 2025-12-07 21:25:30.931848 a 2025-12-14 17:59:06.592191034
  - Columnas telemetría: ['temperature_c', 'humidity_pct', 'battery_pct', 'signal_strength']


,event_id,order_id,timestamp,status,lat,lon,temperature_c,humidity_pct,battery_pct,signal_strength
0,EVT_0000_00,ORD_0000,2025-12-08 21:25:30.923855000,CREATED,19.458318,-98.952440,2.5,63.9,100.0,-51
1,EVT_0000_01,ORD_0000,2025-12-09 00:57:31.889420724,DISPATCHED,19.427410,-98.982584,5.5,79.3,88.9,-46
2,EVT_0000_02,ORD_0000,2025-12-09 00:16:14.887114454,DISPATCHED,19.404439,-99.034971,6.4,75.4,88.5,-52
3,EVT_0000_03,ORD_0000,2025-12-09 06:17:05.819481263,IN_TRANSIT,19.363497,-99.089816,4.2,73.4,65.0,-47
4,EVT_0000_04,ORD_0000,2025-12-09 09:04:33.406617545,IN_TRANSIT,19.345583,-99.123508,2.7,74.3,49.6,-65
5,EVT_0000_05,ORD_0000,2025-12-09 05:57:54.109521443,IN_TRANSIT,19.313996,-99.175062,4.6,64.0,30.2,-42
6,EVT_0000_06,ORD_0000,2025-12-09 09:05:01.191428745,DELIVERED,19.303023,-99.223048,4.5,68.2,24.7,-55
7,EVT_0001_00,ORD_0001,2025-12-09 21:25:30.924854000,CREATED,19.026048,-98.982158,2.6,69.1,100.0,-71
8,EVT_0001_01,ORD_0001,2025-12-10 01:06:10.562037691,DISPATCHED,19.095262,-99.063166,7.4,66.4,93.9,-64
9,EVT_0001_02,ORD_0001,2025-12-10 01:59:16.453088327,IN_TRANSIT,19.125329,-99.136148,2.0,70.2,81.7,-54


---

## 🌡️ Paso 2: Análisis de Telemetría de Sensores

**Análisis de datos de sensores IoT**: Temperatura, humedad, batería y señal por orden.

In [21]:
# Seleccionar una orden de ejemplo para análisis detallado
sample_order = df_transport['order_id'].iloc[0]
df_sample = df_transport[df_transport['order_id'] == sample_order].sort_values('timestamp').copy()

print(f"📦 Analizando Orden: {sample_order}")
print(f"   - Eventos: {len(df_sample)}")
print(f"   - Estados registrados: {df_sample['status'].unique().tolist()}")
print(f"   - Duración: {(df_sample['timestamp'].max() - df_sample['timestamp'].min()).total_seconds()/3600:.1f} horas")

# Añadir tiempo transcurrido desde inicio
df_sample['hours_elapsed'] = (df_sample['timestamp'] - df_sample['timestamp'].min()).dt.total_seconds() / 3600

# Mostrar telemetría completa
print("\n📊 Telemetría de Sensores:")
display(df_sample[['event_id', 'status', 'timestamp', 'temperature_c', 'humidity_pct', 'battery_pct', 'signal_strength']])

# Visualización de telemetría en el tiempo
fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=('Temperatura (°C)', 'Batería (%)', 'Señal (dBm)'),
    vertical_spacing=0.12,
    shared_xaxes=True
)

# Temperatura
fig.add_trace(
    go.Scatter(x=df_sample['hours_elapsed'], y=df_sample['temperature_c'],
              mode='lines+markers', name='Temperatura',
              line=dict(color='red', width=2),
              marker=dict(size=8)),
    row=1, col=1
)
fig.add_hline(y=2, line_dash="dash", line_color="blue", row=1, col=1, 
              annotation_text="Min seguro (2°C)")
fig.add_hline(y=8, line_dash="dash", line_color="orange", row=1, col=1,
              annotation_text="Max seguro (8°C)")

# Batería
fig.add_trace(
    go.Scatter(x=df_sample['hours_elapsed'], y=df_sample['battery_pct'],
              mode='lines+markers', name='Batería',
              line=dict(color='green', width=2),
              marker=dict(size=8)),
    row=2, col=1
)
fig.add_hline(y=20, line_dash="dash", line_color="red", row=2, col=1,
              annotation_text="Nivel crítico (20%)")

# Señal
fig.add_trace(
    go.Scatter(x=df_sample['hours_elapsed'], y=df_sample['signal_strength'],
              mode='lines+markers', name='Señal',
              line=dict(color='purple', width=2),
              marker=dict(size=8)),
    row=3, col=1
)

fig.update_xaxes(title_text="Horas desde inicio", row=3, col=1)
fig.update_yaxes(title_text="°C", row=1, col=1)
fig.update_yaxes(title_text="%", row=2, col=1)
fig.update_yaxes(title_text="dBm", row=3, col=1)

fig.update_layout(height=700, title_text=f"Telemetría IoT - Orden {sample_order}", showlegend=False)
fig.show()

# Estadísticas de la orden
print(f"\n📊 Estadísticas de Telemetría:")
print(f"   Temperatura: {df_sample['temperature_c'].min():.1f}°C - {df_sample['temperature_c'].max():.1f}°C (promedio: {df_sample['temperature_c'].mean():.1f}°C)")
print(f"   Humedad: {df_sample['humidity_pct'].min():.1f}% - {df_sample['humidity_pct'].max():.1f}% (promedio: {df_sample['humidity_pct'].mean():.1f}%)")
print(f"   Batería inicial: {df_sample['battery_pct'].iloc[0]:.1f}% → final: {df_sample['battery_pct'].iloc[-1]:.1f}%")
print(f"   Señal promedio: {df_sample['signal_strength'].mean():.1f} dBm")

📦 Analizando Orden: ORD_0000
   - Eventos: 7
   - Estados registrados: ['CREATED', 'DISPATCHED', 'IN_TRANSIT', 'DELIVERED']
   - Duración: 11.7 horas

📊 Telemetría de Sensores:


,event_id,status,timestamp,temperature_c,humidity_pct,battery_pct,signal_strength
0,EVT_0000_00,CREATED,2025-12-08 21:25:30.923855000,2.5,63.9,100.0,-51
2,EVT_0000_02,DISPATCHED,2025-12-09 00:16:14.887114454,6.4,75.4,88.5,-52
1,EVT_0000_01,DISPATCHED,2025-12-09 00:57:31.889420724,5.5,79.3,88.9,-46
5,EVT_0000_05,IN_TRANSIT,2025-12-09 05:57:54.109521443,4.6,64.0,30.2,-42
3,EVT_0000_03,IN_TRANSIT,2025-12-09 06:17:05.819481263,4.2,73.4,65.0,-47
4,EVT_0000_04,IN_TRANSIT,2025-12-09 09:04:33.406617545,2.7,74.3,49.6,-65
6,EVT_0000_06,DELIVERED,2025-12-09 09:05:01.191428745,4.5,68.2,24.7,-55



📊 Estadísticas de Telemetría:
   Temperatura: 2.5°C - 6.4°C (promedio: 4.3°C)
   Humedad: 63.9% - 79.3% (promedio: 71.2%)
   Batería inicial: 100.0% → final: 24.7%
   Señal promedio: -51.1 dBm


---

## 📊 Paso 4: Resumen de Órdenes en Tránsito

---

## 🚨 Paso 3: Detección de Alertas de Sensores

**Identificación de eventos fuera de rango**: Excursiones de temperatura y problemas de conectividad.

In [22]:
# Detectar eventos con alertas de sensores
print("🚨 Detección de Alertas de Sensores\n")

# Definir umbrales realistas para cadena de frío
TEMP_MIN = 2  # °C
TEMP_MAX = 8  # °C
BATTERY_CRITICAL = 20  # %
SIGNAL_WEAK = -85  # dBm

# Identificar alertas
df_transport['temp_alert'] = (df_transport['temperature_c'] < TEMP_MIN) | (df_transport['temperature_c'] > TEMP_MAX)
df_transport['battery_alert'] = df_transport['battery_pct'] < BATTERY_CRITICAL
df_transport['signal_alert'] = df_transport['signal_strength'] < SIGNAL_WEAK
df_transport['any_alert'] = df_transport['temp_alert'] | df_transport['battery_alert'] | df_transport['signal_alert']

# Clasificar severidad de temperatura
def classify_temp_severity(temp):
    if temp < TEMP_MIN - 2 or temp > TEMP_MAX + 2:
        return 'Crítica'
    elif temp < TEMP_MIN or temp > TEMP_MAX:
        return 'Moderada'
    else:
        return 'Normal'

df_transport['temp_severity'] = df_transport['temperature_c'].apply(classify_temp_severity)

# Estadísticas de alertas
total_events = len(df_transport)
temp_alerts = df_transport['temp_alert'].sum()
battery_alerts = df_transport['battery_alert'].sum()
signal_alerts = df_transport['signal_alert'].sum()
any_alert_count = df_transport['any_alert'].sum()

print("📊 Resumen de Alertas:")
print(f"  Total eventos: {total_events:,}")
print(f"  Eventos con alertas: {any_alert_count} ({any_alert_count/total_events*100:.1f}%)")
print(f"  - Temperatura fuera de rango: {temp_alerts} ({temp_alerts/total_events*100:.1f}%)")
print(f"  - Batería crítica: {battery_alerts} ({battery_alerts/total_events*100:.1f}%)")
print(f"  - Señal débil: {signal_alerts} ({signal_alerts/total_events*100:.1f}%)")

# Órdenes con alertas
orders_with_alerts = df_transport[df_transport['any_alert']].groupby('order_id').size().sort_values(ascending=False)
print(f"\n🚨 Órdenes con alertas: {len(orders_with_alerts)} de {df_transport['order_id'].nunique()}")

# Mostrar eventos críticos
critical_events = df_transport[df_transport['any_alert']].copy()
critical_events['alert_types'] = critical_events.apply(
    lambda row: ', '.join([
        'TEMP' if row['temp_alert'] else '',
        'BATERÍA' if row['battery_alert'] else '',
        'SEÑAL' if row['signal_alert'] else ''
    ]).strip(', '), axis=1
)

print(f"\n⚠️ Eventos Críticos (primeros 10):")
display(critical_events[['event_id', 'order_id', 'timestamp', 'status', 
                         'temperature_c', 'battery_pct', 'signal_strength', 
                         'alert_types']].head(10))

# Visualización de distribución de severidad
severity_counts = df_transport['temp_severity'].value_counts()
fig = px.pie(
    values=severity_counts.values,
    names=severity_counts.index,
    title='Distribución de Severidad de Temperatura',
    color_discrete_map={'Normal': 'green', 'Moderada': 'orange', 'Crítica': 'red'}
)
fig.update_layout(height=400)
fig.show()

print(f"\n✅ Análisis de alertas completado")

🚨 Detección de Alertas de Sensores

📊 Resumen de Alertas:
  Total eventos: 270
  Eventos con alertas: 29 (10.7%)
  - Temperatura fuera de rango: 0 (0.0%)
  - Batería crítica: 0 (0.0%)
  - Señal débil: 29 (10.7%)

🚨 Órdenes con alertas: 23 de 50

⚠️ Eventos Críticos (primeros 10):


,event_id,order_id,timestamp,status,temperature_c,battery_pct,signal_strength,alert_types
13,EVT_0002_01,ORD_0002,2025-12-12 00:05:02.170215970,IN_TRANSIT,4.2,87.0,-86,SEÑAL
17,EVT_0003_02,ORD_0003,2025-12-10 04:41:21.455884029,IN_TRANSIT,6.9,79.4,-89,SEÑAL
26,EVT_0005_01,ORD_0005,2025-12-11 22:59:52.765993423,DISPATCHED,4.2,86.6,-88,SEÑAL
27,EVT_0005_02,ORD_0005,2025-12-12 02:50:44.248130930,IN_TRANSIT,7.9,79.6,-89,SEÑAL
34,EVT_0006_03,ORD_0006,2025-12-13 07:07:17.909487171,IN_TRANSIT,2.5,64.3,-90,SEÑAL
52,EVT_0010_01,ORD_0010,2025-12-13 00:32:59.286763298,IN_TRANSIT,6.2,91.7,-88,SEÑAL
56,EVT_0011_02,ORD_0011,2025-12-14 02:44:54.182394068,IN_TRANSIT,5.8,80.9,-88,SEÑAL
63,EVT_0012_04,ORD_0012,2025-12-10 10:30:36.837779234,IN_TRANSIT,3.2,69.0,-87,SEÑAL
75,EVT_0014_02,ORD_0014,2025-12-12 03:59:05.967685051,IN_TRANSIT,2.4,87.3,-88,SEÑAL
81,EVT_0015_05,ORD_0015,2025-12-08 06:06:52.649180763,IN_TRANSIT,3.7,30.2,-88,SEÑAL



✅ Análisis de alertas completado


In [23]:
# Análisis de eventos por orden
status_order_summary = df_transport.groupby('order_id').agg({
    'event_id': 'count',
    'status': lambda x: x.nunique(),
    'timestamp': ['min', 'max']
}).reset_index()

status_order_summary.columns = ['order_id', 'total_events', 'unique_statuses', 'first_event', 'last_event']

# Calcular duración del viaje
status_order_summary['duration_hours'] = (
    (status_order_summary['last_event'] - status_order_summary['first_event']).dt.total_seconds() / 3600
)

# Filtrar órdenes con múltiples eventos
status_order_summary = status_order_summary[status_order_summary['total_events'] > 1].sort_values('duration_hours', ascending=False)

print("📊 Resumen de Órdenes en Tránsito:")
display(status_order_summary.head(10))

print(f"\n✅ Órdenes analizadas: {len(status_order_summary)}")
print(f"   - Duración promedio: {status_order_summary['duration_hours'].mean():.1f} horas")
print(f"   - Máxima duración: {status_order_summary['duration_hours'].max():.1f} horas")

📊 Resumen de Órdenes en Tránsito:


,order_id,total_events,unique_statuses,first_event,last_event,duration_hours
44,ORD_0044,8,4,2025-12-12 21:25:30.944890,2025-12-14 01:06:11.977198052,27.678065
21,ORD_0021,8,4,2025-12-09 21:25:30.934364,2025-12-10 23:10:16.915721419,25.746106
13,ORD_0013,8,4,2025-12-08 21:25:30.930852,2025-12-09 20:56:28.481484702,23.515986
41,ORD_0041,8,4,2025-12-12 21:25:30.943877,2025-12-13 20:51:42.764557257,23.436617
47,ORD_0047,8,4,2025-12-10 21:25:30.946897,2025-12-11 20:28:18.878902469,23.046648
27,ORD_0027,8,4,2025-12-09 21:25:30.936367,2025-12-10 19:18:03.177406656,21.875623
46,ORD_0046,8,4,2025-12-12 21:25:30.945888,2025-12-13 19:04:34.845328517,21.651083
42,ORD_0042,7,4,2025-12-13 21:25:30.943877,2025-12-14 17:59:06.592191034,20.559902
40,ORD_0040,6,4,2025-12-08 21:25:30.941364,2025-12-09 16:33:39.840395520,19.135805
45,ORD_0045,7,4,2025-12-10 21:25:30.945888,2025-12-11 16:25:21.142683537,18.997277



✅ Órdenes analizadas: 50
   - Duración promedio: 12.4 horas
   - Máxima duración: 27.7 horas


---

## 📈 Paso 5: Distribución de Estados y Análisis Temporal

In [24]:
# Análisis de distribución de eventos por estado
status_distribution = df_transport['status'].value_counts().reset_index()
status_distribution.columns = ['status', 'count']

print("📈 Distribución de Estados de Órdenes:")
display(status_distribution)

# Visualización de estados
fig = px.bar(
    status_distribution,
    x='status',
    y='count',
    title='Distribución de Estados de Órdenes en Tránsito',
    labels={'status': 'Estado', 'count': 'Cantidad de Eventos'},
    color='count',
    color_continuous_scale='viridis'
)
fig.update_layout(height=400)
fig.show()

# Análisis temporal
events_by_day = df_transport.set_index('timestamp').resample('D')['event_id'].count()

print("\n📅 Eventos por Día:")
fig = px.line(
    x=events_by_day.index,
    y=events_by_day.values,
    title='Eventos de Transporte a lo Largo del Tiempo',
    labels={'x': 'Fecha', 'y': 'Cantidad de Eventos'},
    markers=True
)
fig.update_layout(height=400)
fig.show()

📈 Distribución de Estados de Órdenes:


,status,count
0,IN_TRANSIT,133
1,DISPATCHED,56
2,CREATED,50
3,DELIVERED,31



📅 Eventos por Día:


---

## 🗺️ Paso 6: Trazabilidad Geográfica GPS

Visualizar la ruta geográfica de un shipment usando coordenadas GPS de sensores IoT.

In [25]:
# Visualización geográfica de rutas con datos de sensores
print("🗺️ Visualizando Trazabilidad GPS de Red de Distribución:\n")

# Seleccionar una orden para visualización (preferir una con alertas)
orders_with_issues = df_transport[df_transport['any_alert']]['order_id'].unique()
if len(orders_with_issues) > 0:
    sample_order_map = orders_with_issues[0]
    print(f"📦 Orden seleccionada: {sample_order_map} (tiene alertas de sensores)")
else:
    sample_order_map = df_transport['order_id'].iloc[0]
    print(f"📦 Orden seleccionada: {sample_order_map}")

df_map = df_transport[df_transport['order_id'] == sample_order_map].sort_values('timestamp').copy()

print(f"   Puntos de ubicación: {len(df_map)}")
print(f"   Estados: {df_map['status'].unique().tolist()}")
print(f"   Duración: {(df_map['timestamp'].max() - df_map['timestamp'].min()).total_seconds()/3600:.1f} horas")

# Crear columna para hover text con telemetría
df_map['hover_text'] = df_map.apply(
    lambda row: f"Estado: {row['status']}<br>" +
                f"Temp: {row['temperature_c']:.1f}°C<br>" +
                f"Batería: {row['battery_pct']:.0f}%<br>" +
                f"Señal: {row['signal_strength']:.0f} dBm<br>" +
                f"Hora: {row['timestamp'].strftime('%H:%M')}",
    axis=1
)

# Mapa de ruta con código de color por alerta
df_map['marker_color'] = df_map.apply(
    lambda row: 'red' if row['temp_alert'] else 
                'orange' if row['battery_alert'] or row['signal_alert'] else 
                'green',
    axis=1
)

df_map['marker_size'] = df_map.apply(
    lambda row: 12 if row['any_alert'] else 8,
    axis=1
)

fig_map = px.scatter_mapbox(
    df_map,
    lat='lat',
    lon='lon',
    hover_name='hover_text',
    color='marker_color',
    size='marker_size',
    title=f"Trazabilidad GPS con Alertas - Orden {sample_order_map}",
    zoom=10,
    height=550,
    color_discrete_map={'red': 'red', 'orange': 'orange', 'green': 'green'}
)

# Agregar línea de ruta
fig_map.add_trace(go.Scattermapbox(
    lat=df_map['lat'],
    lon=df_map['lon'],
    mode='lines',
    line=dict(width=2, color='blue'),
    name='Ruta',
    showlegend=True
))

fig_map.update_layout(
    mapbox_style="open-street-map",
    legend=dict(
        title="Leyenda",
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)
fig_map.show()

print(f"\n✅ Ruta mapeada con {len(df_map)} eventos")
if df_map['any_alert'].sum() > 0:
    print(f"⚠️ {df_map['any_alert'].sum()} eventos con alertas detectadas en esta ruta")

🗺️ Visualizando Trazabilidad GPS de Red de Distribución:

📦 Orden seleccionada: ORD_0002 (tiene alertas de sensores)
   Puntos de ubicación: 3
   Estados: ['CREATED', 'IN_TRANSIT']
   Duración: 5.8 horas



✅ Ruta mapeada con 3 eventos
⚠️ 1 eventos con alertas detectadas en esta ruta


---

## 📊 Paso 7: Distribución de Eventos por Estado

In [26]:
# Análisis de eventos por estado y región
print("📊 Estadísticas de Eventos por Estado:\n")

# Conteo de eventos por estado
status_counts = df_transport['status'].value_counts()
print(status_counts)

# Gráfico de distribución de eventos por estado
fig = px.bar(
    status_counts.reset_index(),
    x='status',
    y='count',
    color='status',
    title="Distribución de Eventos por Estado de Orden",
    labels={'status': 'Estado', 'count': 'Cantidad de Eventos'},
    color_discrete_sequence=['#0066CC', '#FF9900', '#FFCC00', '#00CC00']
)

fig.update_layout(
    xaxis_title="Estado de Orden",
    yaxis_title="Cantidad de Eventos",
    showlegend=False,
    height=400
)
fig.show()

print("\n✅ Distribución de eventos analizada")

📊 Estadísticas de Eventos por Estado:

status
IN_TRANSIT    133
DISPATCHED     56
CREATED        50
DELIVERED      31
Name: count, dtype: int64



✅ Distribución de eventos analizada


---

## 🔍 Paso 8: Análisis de Órdenes con Mayor Actividad

In [27]:
# Análisis de eventos por orden (equivalente a "eventos críticos")
print("🔍 Análisis de Órdenes con Mayor Cantidad de Eventos:\n")

# Contar eventos por orden
order_event_summary = df_transport.groupby('order_id').agg({
    'event_id': 'count',
    'status': 'nunique',
    'timestamp': ['min', 'max']
}).round(2)

order_event_summary.columns = ['num_eventos', 'num_estados', 'timestamp_inicio', 'timestamp_fin']
order_event_summary = order_event_summary.sort_values('num_eventos', ascending=False)

print(f"📦 Órdenes con Mayor Actividad:")
display(order_event_summary.head(20))

# Guardar reporte
output_dir = Path('data/processed')
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "iot_order_events_summary.csv"
order_event_summary.to_csv(output_file)
print(f"\n💾 Reporte guardado: {output_file}")

# Estadísticas generales
print(f"\n📊 Estadísticas de Eventos por Orden:")
print(f"  Promedio de eventos/orden: {order_event_summary['num_eventos'].mean():.1f}")
print(f"  Máximo de eventos/orden: {order_event_summary['num_eventos'].max():.0f}")
print(f"  Mínimo de eventos/orden: {order_event_summary['num_eventos'].min():.0f}")

🔍 Análisis de Órdenes con Mayor Cantidad de Eventos:

📦 Órdenes con Mayor Actividad:


,num_eventos,num_estados,timestamp_inicio,timestamp_fin
order_id,,,,
ORD_0013,8,4,2025-12-08 21:25:30.930852,2025-12-09 20:56:28.481484702
ORD_0015,8,4,2025-12-07 21:25:30.931848,2025-12-08 12:51:59.269892152
ORD_0035,8,4,2025-12-07 21:25:30.940365,2025-12-08 15:36:09.007763365
ORD_0037,8,4,2025-12-11 21:25:30.941364,2025-12-12 13:20:25.214708844
ORD_0041,8,4,2025-12-12 21:25:30.943877,2025-12-13 20:51:42.764557257
ORD_0044,8,4,2025-12-12 21:25:30.944890,2025-12-14 01:06:11.977198052
ORD_0046,8,4,2025-12-12 21:25:30.945888,2025-12-13 19:04:34.845328517
ORD_0033,8,4,2025-12-07 21:25:30.939365,2025-12-08 11:58:08.982091622
ORD_0027,8,4,2025-12-09 21:25:30.936367,2025-12-10 19:18:03.177406656



💾 Reporte guardado: data\processed\iot_order_events_summary.csv

📊 Estadísticas de Eventos por Orden:
  Promedio de eventos/orden: 5.4
  Máximo de eventos/orden: 8
  Mínimo de eventos/orden: 3


---

## 📊 Paso 9: Dashboard de KPIs de Monitoreo IoT

In [28]:
# Calcular KPIs completos de monitoreo IoT
print("📊 KPIs de Monitoreo IoT - Red de Distribución\n")

# KPIs operativos
total_orders = df_transport['order_id'].nunique()
total_events = len(df_transport)
avg_events_per_order = total_events / total_orders
orders_delivered = (df_transport.groupby('order_id')['status'].apply(lambda x: 'DELIVERED' in x.values)).sum()
delivery_rate = (orders_delivered / total_orders * 100) if total_orders > 0 else 0

# KPIs de calidad de sensores
temp_compliance = ((~df_transport['temp_alert']).sum() / total_events * 100)
battery_health = ((~df_transport['battery_alert']).sum() / total_events * 100)
signal_quality = ((~df_transport['signal_alert']).sum() / total_events * 100)
overall_compliance = ((~df_transport['any_alert']).sum() / total_events * 100)

# Contar órdenes en cada estado (última actualización)
df_latest_status = df_transport.sort_values('timestamp').drop_duplicates('order_id', keep='last')
status_distribution = df_latest_status['status'].value_counts()

# Dashboard de KPIs con 6 indicadores
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=(
        'Delivery Rate', 'Eventos/Orden', 'Compliance Total',
        'Temp. Compliance', 'Batería OK', 'Señal OK'
    ),
    specs=[[{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}],
           [{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}]]
)

# Fila 1: KPIs operativos
fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=delivery_rate,
    title={'text': "Delivery Rate (%)"},
    gauge={
        'axis': {'range': [None, 100]},
        'bar': {'color': "green" if delivery_rate >= 90 else "orange"},
        'threshold': {'line': {'color': "red", 'width': 3}, 'thickness': 0.75, 'value': 90}
    }
), row=1, col=1)

fig.add_trace(go.Indicator(
    mode="number+delta",
    value=avg_events_per_order,
    title={'text': "Eventos Prom/Orden"},
    delta={'reference': 5, 'relative': False}
), row=1, col=2)

fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=overall_compliance,
    title={'text': "Compliance Total (%)"},
    gauge={
        'axis': {'range': [None, 100]},
        'bar': {'color': "green" if overall_compliance >= 85 else "red"},
        'threshold': {'line': {'color': "red", 'width': 3}, 'thickness': 0.75, 'value': 85}
    }
), row=1, col=3)

# Fila 2: KPIs de sensores
fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=temp_compliance,
    title={'text': "Temp. Compliance (%)"},
    gauge={
        'axis': {'range': [None, 100]},
        'bar': {'color': "green" if temp_compliance >= 95 else "orange"},
        'threshold': {'line': {'color': "red", 'width': 2}, 'thickness': 0.75, 'value': 95}
    }
), row=2, col=1)

fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=battery_health,
    title={'text': "Batería OK (%)"},
    gauge={
        'axis': {'range': [None, 100]},
        'bar': {'color': "green" if battery_health >= 90 else "red"}
    }
), row=2, col=2)

fig.add_trace(go.Indicator(
    mode="gauge+number",
    value=signal_quality,
    title={'text': "Señal OK (%)"},
    gauge={
        'axis': {'range': [None, 100]},
        'bar': {'color': "green" if signal_quality >= 80 else "orange"}
    }
), row=2, col=3)

fig.update_layout(
    title="Dashboard KPIs IoT - Monitoreo de Flota y Sensores",
    height=600,
    showlegend=False
)
fig.show()

# Resumen ejecutivo
print("\n📋 RESUMEN EJECUTIVO - MONITOREO IoT")
print("="*70)
print(f"\n🚚 OPERACIÓN:")
print(f"  Total de Órdenes: {total_orders}")
print(f"  Total de Eventos IoT: {total_events:,}")
print(f"  Eventos Promedio/Orden: {avg_events_per_order:.2f}")
print(f"  Delivery Rate: {delivery_rate:.1f}%")
print(f"  Órdenes Entregadas: {orders_delivered}/{total_orders}")

print(f"\n📡 CALIDAD DE SENSORES:")
print(f"  Compliance Total: {overall_compliance:.1f}%")
print(f"  Temperatura en Rango: {temp_compliance:.1f}%")
print(f"  Batería Saludable: {battery_health:.1f}%")
print(f"  Señal Adecuada: {signal_quality:.1f}%")

print(f"\n⚠️ ALERTAS:")
print(f"  Eventos con alertas: {df_transport['any_alert'].sum()} ({df_transport['any_alert'].sum()/total_events*100:.1f}%)")
print(f"  - Temperatura: {df_transport['temp_alert'].sum()}")
print(f"  - Batería: {df_transport['battery_alert'].sum()}")
print(f"  - Señal: {df_transport['signal_alert'].sum()}")

print(f"\n📊 DISTRIBUCIÓN POR ESTADO (última actualización):")
for status, count in status_distribution.items():
    print(f"  {status}: {count} órdenes ({count/total_orders*100:.1f}%)")

📊 KPIs de Monitoreo IoT - Red de Distribución




📋 RESUMEN EJECUTIVO - MONITOREO IoT

🚚 OPERACIÓN:
  Total de Órdenes: 50
  Total de Eventos IoT: 270
  Eventos Promedio/Orden: 5.40
  Delivery Rate: 62.0%
  Órdenes Entregadas: 31/50

📡 CALIDAD DE SENSORES:
  Compliance Total: 89.3%
  Temperatura en Rango: 100.0%
  Batería Saludable: 100.0%
  Señal Adecuada: 89.3%

⚠️ ALERTAS:
  Eventos con alertas: 29 (10.7%)
  - Temperatura: 0
  - Batería: 0
  - Señal: 29

📊 DISTRIBUCIÓN POR ESTADO (última actualización):
  IN_TRANSIT: 33 órdenes (66.0%)
  DELIVERED: 17 órdenes (34.0%)


---

# 📤 SECCIONES FINALES

---

## 💾 Exportar Resultados

In [29]:
# Exportar resultados procesados con métricas IoT completas
processed_path = OUTPUT_DIR / 'rt04_iot_sensors'
processed_path.mkdir(parents=True, exist_ok=True)

# 1. Dataset completo con alertas
df_transport.to_csv(processed_path / 'transport_events_with_alerts.csv', index=False)

# 2. Resumen de eventos por orden
order_event_summary.to_csv(processed_path / 'order_events_summary.csv', index=True)

# 3. Distribución de estados
pd.DataFrame({
    'status': status_distribution.index,
    'count': status_distribution.values,
    'percentage': (status_distribution.values / total_orders * 100).round(2)
}).to_csv(processed_path / 'status_distribution.csv', index=False)

# 4. KPIs completos
kpis_summary = pd.DataFrame({
    'categoria': ['Operación', 'Operación', 'Operación', 'Operación', 
                  'Sensores', 'Sensores', 'Sensores', 'Sensores'],
    'metrica': ['Total Órdenes', 'Total Eventos', 'Eventos Prom/Orden', 'Delivery Rate (%)',
                'Compliance Total (%)', 'Temp. Compliance (%)', 'Batería OK (%)', 'Señal OK (%)'],
    'valor': [total_orders, total_events, round(avg_events_per_order, 2), round(delivery_rate, 2),
              round(overall_compliance, 2), round(temp_compliance, 2), round(battery_health, 2), round(signal_quality, 2)]
})
kpis_summary.to_csv(processed_path / 'kpis_monitoreo_iot.csv', index=False)

# 5. Resumen de alertas
alerts_summary = pd.DataFrame({
    'tipo_alerta': ['Temperatura', 'Batería', 'Señal', 'Cualquier alerta'],
    'cantidad': [df_transport['temp_alert'].sum(), 
                 df_transport['battery_alert'].sum(),
                 df_transport['signal_alert'].sum(),
                 df_transport['any_alert'].sum()],
    'porcentaje': [
        round(df_transport['temp_alert'].sum()/total_events*100, 2),
        round(df_transport['battery_alert'].sum()/total_events*100, 2),
        round(df_transport['signal_alert'].sum()/total_events*100, 2),
        round(df_transport['any_alert'].sum()/total_events*100, 2)
    ]
})
alerts_summary.to_csv(processed_path / 'alerts_summary.csv', index=False)

# 6. Eventos críticos
critical_events = df_transport[df_transport['any_alert']].copy()
critical_events.to_csv(processed_path / 'critical_events.csv', index=False)

print(f"✅ Resultados exportados a {processed_path}")
print(f"   Archivos generados:")
print(f"   - transport_events_with_alerts.csv ({len(df_transport):,} registros)")
print(f"   - order_events_summary.csv ({len(order_event_summary)} órdenes)")
print(f"   - status_distribution.csv")
print(f"   - kpis_monitoreo_iot.csv (8 KPIs)")
print(f"   - alerts_summary.csv")
print(f"   - critical_events.csv ({len(critical_events)} eventos críticos)")

✅ Resultados exportados a f:\GitHub\supply-chain-data-notebooks\data\processed\rt04_iot_sensors
   Archivos generados:
   - transport_events_with_alerts.csv (270 registros)
   - order_events_summary.csv (50 órdenes)
   - status_distribution.csv
   - kpis_monitoreo_iot.csv (8 KPIs)
   - alerts_summary.csv
   - critical_events.csv (29 eventos críticos)


---

## ✅ Validaciones

In [30]:
# Validaciones de integridad y lógica de negocio IoT
print("🔍 Ejecutando validaciones de integridad...\n")

# Validaciones de datos base
assert len(df_transport) > 0, "El dataset de eventos no debe estar vacío"
assert df_transport['order_id'].notna().all(), "No deben existir order_id nulos"
assert df_transport['timestamp'].notna().all(), "No deben existir timestamps nulos"
assert total_orders > 0, "Debe haber al menos una orden procesada"

# Validaciones de KPIs operativos
assert 0 <= delivery_rate <= 100, "El delivery rate debe estar entre 0 y 100%"
assert avg_events_per_order >= 1, "Cada orden debe tener al menos un evento"

# Validaciones de telemetría de sensores
assert df_transport['temperature_c'].notna().all(), "No deben existir valores nulos de temperatura"
assert df_transport['humidity_pct'].notna().all(), "No deben existir valores nulos de humedad"
assert df_transport['battery_pct'].notna().all(), "No deben existir valores nulos de batería"
assert df_transport['signal_strength'].notna().all(), "No deben existir valores nulos de señal"

# Validaciones de rangos realistas
assert df_transport['temperature_c'].min() >= -20, "Temperatura mínima fuera de rango realista"
assert df_transport['temperature_c'].max() <= 50, "Temperatura máxima fuera de rango realista"
assert df_transport['humidity_pct'].min() >= 0, "Humedad debe ser no negativa"
assert df_transport['humidity_pct'].max() <= 100, "Humedad no puede exceder 100%"
assert df_transport['battery_pct'].min() >= 0, "Batería debe ser no negativa"
assert df_transport['battery_pct'].max() <= 100, "Batería no puede exceder 100%"
assert df_transport['signal_strength'].min() >= -120, "Señal fuera de rango típico"
assert df_transport['signal_strength'].max() <= 0, "Señal no puede ser positiva (dBm)"

# Validaciones de compliance
assert 0 <= temp_compliance <= 100, "Compliance de temperatura debe estar entre 0 y 100%"
assert 0 <= overall_compliance <= 100, "Compliance total debe estar entre 0 y 100%"

print("✅ Todas las validaciones pasadas correctamente")
print(f"✅ Notebook RT-04 completado: Análisis exploratorio de {total_events:,} eventos IoT")
print(f"   - {total_orders} órdenes analizadas")
print(f"   - {df_transport['any_alert'].sum()} eventos con alertas detectados")
print(f"   - Compliance total: {overall_compliance:.1f}%")

🔍 Ejecutando validaciones de integridad...

✅ Todas las validaciones pasadas correctamente
✅ Notebook RT-04 completado: Análisis exploratorio de 270 eventos IoT
   - 50 órdenes analizadas
   - 29 eventos con alertas detectados
   - Compliance total: 89.3%


---

## 📚 Resumen Técnico y Referencias

### 🎯 Resultados Clave

Este análisis implementa un sistema completo de monitoreo IoT de sensores de flota para logística con detección de alertas en tiempo real.

**Métricas calculadas:**
1. **Delivery Rate**: `(Órdenes Entregadas / Total Órdenes) × 100` - Mide el porcentaje de entregas exitosas
2. **Eventos Promedio/Orden**: `Total Eventos / Total Órdenes` - Indica la trazabilidad y nivel de monitoreo
3. **Temperature Compliance**: `(Eventos sin alerta de temp / Total Eventos) × 100` - Cumplimiento de cadena de frío (2-8°C)
4. **Battery Health**: `(Eventos con batería >20% / Total Eventos) × 100` - Salud de dispositivos IoT
5. **Signal Quality**: `(Eventos con señal >-85dBm / Total Eventos) × 100` - Calidad de conectividad
6. **Overall Compliance**: `(Eventos sin alertas / Total Eventos) × 100` - Compliance total del sistema

**Hallazgos típicos:**
- Patrones temporales revelan picos de actividad logística (horarios específicos)
- Trazabilidad GPS permite correlacionar ubicación con eventos de temperatura
- Alertas de batería predicen fallos de dispositivos antes de ocurrir
- Análisis de señal identifica zonas de cobertura deficiente
- Excursiones de temperatura ocurren principalmente durante carga/descarga

**Visualizaciones clave:**
- **Gráficos de telemetría multi-sensor**: Temperatura, batería y señal en línea temporal
- **Mapas interactivos con alertas**: Trazabilidad geográfica con código de color por severidad
- **Dashboard de 6 KPIs**: Operación (delivery, eventos) + Sensores (temp, batería, señal)
- **Gráficos de distribución**: Severidad de temperatura, estados de órdenes

### 🔬 Metodología

**Técnica principal: Análisis Exploratorio de Datos IoT con Detección de Alertas**

**Fórmulas de Compliance:**

$$
\text{Temperature Compliance} = \frac{\sum(\text{TEMP\_MIN} \leq T \leq \text{TEMP\_MAX})}{\text{Total Eventos}} \times 100
$$

$$
\text{Battery Health} = \frac{\sum(\text{Battery} \geq 20\%)}{\text{Total Eventos}} \times 100
$$

$$
\text{Signal Quality} = \frac{\sum(\text{Signal} \geq -85 \text{ dBm})}{\text{Total Eventos}} \times 100
$$

$$
\text{Overall Compliance} = \frac{\text{Eventos sin alertas}}{\text{Total Eventos}} \times 100
$$

**Umbrales realistas aplicados:**
- Temperatura cadena de frío: 2°C - 8°C (estándar farmacéutico/alimenticio)
- Batería crítica: < 20% (permite tiempo de reacción antes de fallo)
- Señal débil: < -85 dBm (umbral típico para conectividad 4G/LTE confiable)

**Herramientas utilizadas:**
- Pandas para agregaciones complejas y análisis temporal
- Plotly para visualización interactiva (subplots, mapbox, gauges)
- NumPy para generación de datos sintéticos realistas con distribuciones apropiadas

### 📖 Aplicaciones Prácticas

1. **Monitoreo de Cadena de Frío:**
   - Dashboard en tiempo real de compliance de temperatura
   - Alertas automáticas SMS/email cuando temperatura sale de rango
   - Reporte de excursiones para auditoría regulatoria (FDA, COFEPRIS)

2. **Mantenimiento Predictivo de Dispositivos:**
   - Predicción de fallos de batería (alertas 24-48h antes)
   - Identificación de dispositivos con deterioro de señal
   - Planificación de reemplazo de sensores antes de fallo crítico

3. **Optimización de Rutas GPS:**
   - Análisis histórico de rutas para planificación óptima
   - Identificación de zonas con problemas de cobertura
   - Correlación de alertas de temperatura con ubicaciones específicas

4. **Fundamentos para Sistemas Avanzados:**
   - Features para ML: Predicción de tiempo de entrega basado en telemetría
   - Stream processing: Implementación con Kafka para alertas en tiempo real (RT-01)
   - Clasificación de riesgo: Modelos para predecir órdenes problemáticas

### 🔗 Referencias

1. **FDA, 2023**. *Guidance for Industry: Good Distribution Practices for Medical Devices*. U.S. Food and Drug Administration.
   - Estándares de cadena de frío y rangos de temperatura para productos sensibles

2. **3GPP, 2022**. *LTE Signal Strength Standards (TS 36.133)*. 3rd Generation Partnership Project.
   - Especificaciones técnicas de calidad de señal celular para IoT

3. **McKinsey & Company, 2022**. *The future of IoT in supply chain management*. McKinsey Digital.
   - Estrategias de implementación de IoT y ROI en logística

4. **IEEE IoT Journal, 2021**. *Real-time tracking systems for cold chain logistics*. Vol. 8, Issue 12.
   - Arquitecturas de referencia para sensores de temperatura en tiempo real

5. **Gartner, 2023**. *IoT in Supply Chain: Technology Trends*. Gartner Research.
   - Tendencias de adopción: LoRaWAN, NB-IoT, edge computing

### 💡 Extensiones Futuras

- **Stream Processing en Tiempo Real**: Apache Kafka + Spark Streaming para procesamiento sub-segundo (ver RT-01)
- **Machine Learning Predictivo**: Modelos de predicción de excursiones de temperatura basados en ruta/clima (ver DS-07)
- **Cold Chain Avanzado**: Análisis específico de cadena de frío farmacéutico con validación regulatoria (ver RT-03)
- **Alertas Multi-Canal**: Sistema de notificaciones con SMS, email, webhook para eventos críticos
- **Integración con Edge Computing**: Procesamiento de alertas en dispositivos (Raspberry Pi, Arduino) sin depender de cloud
- **Digital Twin**: Gemelo digital de la flota para simulación de escenarios y optimización
- **Blockchain para Trazabilidad**: Registro inmutable de eventos de temperatura para auditoría

---

**Autor**: lraigosov (@LuisRai)  
**Fecha**: 2024 a la actualidad  
**Versión**: 2.0  
**Tags**: `#iot` `#sensores` `#telemetría` `#fleet-tracking` `#real-time` `#cold-chain` `#logistics` `#alerts`

---

<div style="width: 100%; clear: both; margin: 0 0 20px 0; border-top: 1px solid #eaecef; padding-top: 24px;"><div style="display: flex; justify-content: space-between; align-items: center; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif;"><div style="flex: 1; text-align: left;"><a href="TR-01-transporte_masivo.ipynb" style="text-decoration: none; color: #0366d6; font-size: 14px; font-weight: 600; transition: color 0.2s;">← Anterior: [TR-01-transporte_masivo.ipynb](../60_realtime_iot/TR-01-transporte_masivo.ipynb)</a></div><div style="flex: 1; text-align: center; font-size: 14px;"><a href="../../README.md" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📑 Índice</a><span style="color: #6a737d;">|</span><a href="../../config/notebooks_index.yml" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📋 Catálogo</a></div><div style="flex: 1; text-align: right;"><span style="color: #6a737d; font-size: 14px; cursor: default;">Siguiente →</span></div></div></div>

